# 单被试运动想象解码 Demo（教学详注版）

端到端走一遍 MI-Decoder 流水线。每个代码块包含三段式注释：
**【方法】**是什么原理、**【为什么选它】**同类方法比较、**【目的】**这一步解决什么问题，
并打印每一步的中间输出，让数据的变化全程可见。

1. 下载与加载数据（PhysioNet EEGBCI，被试 S001）
2. 预处理：坏导检测 → 插值 → 平均参考 → 滤波
3. ICA 去眼电
4. Epoch 切分（按事件对齐试次）
5. ERD 神经生理验证（解码前的质量门禁）
6. CSP + LDA 解码 + 空间模式可视化
7. mini 伪在线：因果滤波下的延迟-精度曲线

> 批量版（20 被试全流程）见项目根目录 README：`python -m src.preprocess` 等。
> 本 notebook 复用 `src/` 中的函数，保证与批量版逻辑一致。

In [ ]:
# ============ 环境准备 ============
# 【目的】把项目根目录加入 sys.path, 才能 import src 包(与批量版共用同一套函数,
#        避免 notebook 和脚本两套逻辑逐渐漂移——工程上叫"单一事实来源")
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt

# 【坑-中文乱码】matplotlib 默认字体 DejaVu Sans 没有中文字形, 图中汉字会变豆腐块;
#        Mac 自带的几款中文字体按优先级依次尝试, unicode_minus 解决负号显示成方块的问题
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "Hiragino Sans GB", "PingFang SC"]
plt.rcParams["axes.unicode_minus"] = False
import mne
import numpy as np
from mne.datasets import eegbci

from src.utils import load_config

# 【说明】MNE 默认输出非常啰嗦, 收窄到 WARNING 只看真正重要的信息
mne.set_log_level("WARNING")

cfg = load_config(PROJECT_ROOT / "config.yaml")
SUBJECT = 1                     # 换被试只改这里

# ---- 打印: 运行环境与本次配置 ----
print(f"MNE 版本: {mne.__version__}")
print(f"被试: S{SUBJECT:03d}")
print(f"预处理配置: {cfg['preprocess']}")
print(f"解码配置:   {cfg['decode']}")

## Step 1 下载与加载

EEGBCI：64 导、160 Hz。runs 4/8/12 是**左右手运动想象**任务，
注释 T1=左手、T2=右手、T0=静息。

📚 **延伸干货**
- 📄 [PhysioNet EEGBCI 数据集主页](https://physionet.org/content/eegmmidb/1.0.0/)——数据采集范式的官方说明
- 📄 [Python脑电处理中文手册](https://github.com/ZitongLu1996/Python-EEG-Handbook-CN)——中文最全的 MNE 实操手册, 可在 Colab 直接跑
- 🎬 [B站: Python脑电数据处理EEG-基于MNE包](https://www.bilibili.com/video/BV1fg411w7Z3)——从环境配置到预处理的系列课

In [ ]:
# ============ Step 1: 下载与加载 + 电极定位 ============
#
# 【方法】EDF 是脑电通用存储格式(欧洲数据格式); 三个 run 是同任务的三段录制,
#        concatenate 拼成一条连续数据, 试次数从 15 提升到 ~45(样本量对小数据至关重要)。
#        set_montage 给每个电极挂上标准头模上的三维坐标。
#
# 【为什么选它】电极坐标有两种来源: ①标准模板(standard_1005, 假设所有人头型一致)
#        ②个体化数字化仪实测(Polhemus, 更准但需要采集时记录)。
#        公开数据集没有实测坐标, 用模板是唯一选择, 对插值/地形图精度足够。
#
# 【目的】坐标是后面三件事的几何基础: 坏导插值(需要知道谁和谁相邻)、
#        地形图(把数值画到头皮位置)、CSP patterns 可视化。

files = eegbci.load_data(SUBJECT, cfg["data"]["runs"])   # 首次运行自动下载(~7MB)
raw = mne.concatenate_raws([mne.io.read_raw_edf(f, preload=True) for f in files])
eegbci.standardize(raw)              # 通道名规范化: 'Fc5.' -> 'FC5'(匹配标准命名)
raw.set_montage("standard_1005")     # 挂三维坐标

# ---- 打印: 数据概况 ----
print(raw)
print(f"通道数: {len(raw.ch_names)} | 采样率: {raw.info['sfreq']} Hz "
      f"| 总时长: {raw.times[-1]:.0f} s")
print(f"前10个通道名: {raw.ch_names[:10]}")
print(f"数据矩阵形状 (通道, 时间点): {raw.get_data().shape}")
print(f"信号幅值范围: [{raw.get_data().min()*1e6:.0f}, "
      f"{raw.get_data().max()*1e6:.0f}] uV")

# 电极布局图: 确认 C3/C4 在中央区两侧(后面所有分析的核心电极)
raw.plot_sensors(show_names=True);

## Step 2 坏导检测 → 插值 → 重参考 → 滤波

顺序不能乱：**插值必须在平均参考之前**——坏导会把参考(所有通道的均值)拉偏，
把一个通道的问题扩散成所有通道的问题。

📚 **延伸干货**
- 📄 [MNE 官方滤波原理长文](https://mne.tools/stable/auto_tutorials/preprocessing/25_background_filtering.html)——零相位 vs 因果、FIR vs IIR 讲得最透的一篇, 强烈推荐精读
- 📄 [知乎: MNE-Python 处理脑电教程汇总](https://zhuanlan.zhihu.com/p/128667251)——中文入口索引
- 🎬 [B站:【Python+MNE】脑电数据分析教程](https://www.bilibili.com/video/BV1zz4y1k7wN)——8集系列, 覆盖预处理全流程

In [ ]:
# ============ Step 2a: 坏导自动检测 ============
#
# 【方法】对每个通道计算整段信号的对数方差, 再算 z 分数(相对中位数的偏离程度)。
#        接触不良的电极方差异常大(纯噪声)或异常小(近乎断连), |z|>3 判为坏导。
#        取对数是因为方差跨通道分布右偏, log 后近似正态, z 分数才有意义。
#
# 【为什么选它】同类方法比较:
#        ①人工目检(实验室金标准)——批量处理不可行, 且不可复现
#        ②pyprep 的 RANSAC(用邻居电极预测该通道, 预测不准判坏)——更严谨但慢一个量级
#        ③相关性法(与邻居的相关系数过低判坏)——对整体漂移不敏感
#        对数方差 z 分数是"够用且极快"的工程折中, 大规模流水线常用。
#
# 【目的】坏导不修, 后面平均参考会被污染、ICA 会浪费成分去拟合它。

from src.preprocess import auto_detect_bads

p = cfg["preprocess"]

# ---- 打印: 每个通道的 z 分数明细(看看检测器眼里的世界) ----
data_ = raw.get_data(picks="eeg")
ch_var = np.log(np.var(data_, axis=1) + 1e-20)
z = (ch_var - np.median(ch_var)) / (np.std(ch_var) + 1e-20)
order = np.argsort(np.abs(z))[::-1]
print("z分数绝对值 Top5 通道(候选坏导):")
for i in order[:5]:
    flag = "  <-- 超阈值!" if abs(z[i]) > p["bad_z_thresh"] else ""
    print(f"  {raw.ch_names[i]:>6s}: z={z[i]:+.2f}{flag}")

raw.info["bads"] = auto_detect_bads(raw, p["bad_z_thresh"])
print(f"\n判定坏导(|z|>{p['bad_z_thresh']}): {raw.info['bads'] or '无'}")

In [ ]:
# ============ Step 2b: 插值坏导 + 平均参考 ============
#
# 【方法-插值】球面样条插值: 把头皮当球面, 用全部好电极按几何距离加权重建坏导信号。
#        物理依据是"容积传导"——脑电经过颅骨扩散后在头皮上空间平滑, 相邻电极信号高度相似。
# 【为什么不直接删掉坏导】①删了通道数就变了, 跨被试没法对齐特征维度;
#        ②平均参考要求全头覆盖均匀, 缺一块参考就偏了。插值保持维度、代价是该通道信息量为零。
#
# 【方法-重参考】脑电测的是"电位差", 必须指定参考点。平均参考 = 每个时刻
#        减去所有通道均值, 近似"无穷远零参考"。
# 【为什么选它】同类比较: ①单侧乳突参考——离一侧近, 引入左右不对称偏差(MI任务致命,
#        因为我们要比的就是左右半球差异!) ②双侧乳突平均——好些但仍受局部影响
#        ③REST(参考电极标准化)——理论最优但需要头模型。
#        64导全头覆盖时平均参考是标准选择(经验法则: >=32导才建议用)。
#
# 【目的】给左右半球一个公平的比较基准——C3 vs C4 的 ERD 差异就是解码特征本身。

if raw.info["bads"]:
    before = raw.get_data(picks=raw.info["bads"])       # 留一份插值前的数据
    raw.interpolate_bads(reset_bads=True)
    print(f"已插值 {len(before)} 个坏导(球面样条重建)")
else:
    print("无坏导, 跳过插值")

# 打印: 平均参考前后, 信号均值的变化(参考做对了, 每时刻跨通道均值应为0)
mean_before = raw.get_data(picks="eeg").mean(axis=0).std() * 1e6
raw.set_eeg_reference("average", projection=False)
mean_after = raw.get_data(picks="eeg").mean(axis=0).std() * 1e6
print(f"跨通道均值的波动幅度: 重参考前 {mean_before:.3f} uV "
      f"-> 重参考后 {mean_after:.6f} uV (应接近0)")

In [ ]:
# ============ Step 2c: 带通滤波 1-45 Hz ============
#
# 【方法】FIR 带通滤波器, MNE 默认零相位(filtfilt: 正向滤一遍再反向滤一遍,
#        相位失真互相抵消)。高通 1Hz 去基线漂移(皮肤出汗/电极极化造成的超低频飘移),
#        低通 45Hz 去肌电和高频噪声(也顺带压掉 50/60Hz 工频)。
#
# 【为什么选它】同类比较:
#        ①FIR vs IIR: FIR 稳定、线性相位, 离线首选; IIR(butter)阶数低、延迟小, 在线用
#        ②零相位 vs 因果: 零相位不扭曲波形形状, 但反向那一遍用了"未来"数据,
#          只允许离线分析用! Step 7 会换成因果滤波演示在线约束。
#        ③高通截止选 1Hz 而非 0.1Hz: 更激进地去漂移, 且显著改善 ICA 分解质量
#          (ICA 对超低频漂移敏感); 代价是丢掉 <1Hz 信息, 对 MI(8-30Hz)无损。
#
# 【目的】把与任务无关的频段物理删除, 给 ICA 和后续分析一个干净的输入。

psd_before = raw.compute_psd(fmax=70)                   # 留一份滤波前的功率谱
raw.filter(l_freq=p["l_freq"], h_freq=p["h_freq"])
psd_after = raw.compute_psd(fmax=70)

# ---- 打印: 滤波前后对比 ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
psd_before.plot(axes=axes[0], show=False)
axes[0].set_title("滤波前: 注意60Hz工频尖峰与低频能量")
psd_after.plot(axes=axes[1], show=False)
axes[1].set_title("滤波后: 45Hz以上被压掉, ~10Hz的mu/alpha峰清晰")
fig.tight_layout()
print(f"滤波配置: {p['l_freq']}-{p['h_freq']} Hz (零相位FIR, 仅限离线!)")

## Step 3 ICA 去眼电

眨眼的电场比脑电大一个量级，且频率(1-10Hz)与脑电重叠——**滤波去不掉**，
必须走空间分离的路线。

📚 **延伸干货**
- 📄 [MNE ICA 去伪迹官方教程](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html)——本步骤代码的权威出处
- 🎬 [YouTube: ICA applied to EEG 播放列表](https://www.youtube.com/playlist?list=PLXc9qfVbMMN2uDadxZ_OEsHjzcRtlLNxc)——EEGLAB 作者 Arnaud Delorme 亲授, 理论+实操
- 🧪 [ICLabel 在线练习](https://labeling.ucsd.edu/tutorial)——练"肉眼认成分"的互动网站, 刷几十个就有手感

In [ ]:
# ============ Step 3: ICA 独立成分分析去眼电 ============
#
# 【方法】ICA 沿"通道轴"做盲源分离: 假设 64 通道信号是若干统计独立的源
#        (脑源+眼电+肌电+心电)的线性混合, 解出解混矩阵把源拆开。
#        注意: 是按"统计独立性"拆, 不是按频率拆! 每个成分仍是宽带信号。
#        眨眼成分的指纹: 地形图能量集中在额区最前部 + 时间上是稀疏大尖峰。
#
# 【为什么选它】同类比较:
#        ①回归法(EOG通道做参考回归掉)——需要专门EOG电极, 且会连带减掉相关脑电
#        ②SSP(信号空间投影)——快, 但投影方向少, 去得不如ICA干净
#        ③ICA——不需要参考电极, 分离最彻底, 预处理标配; 代价是计算慢、需要人工/自动选成分
#        EEGBCI 没有 EOG 通道, 用最靠近眼睛的 Fp1/Fp2 做"代理", 
#        find_bads_eog 找与它们时间相关性最高的成分。
#
# 【目的】眨眼伪迹幅值大且集中在额区, 不除掉的话: ERD地形图会被额区污染,
#        CSP 也可能学到"眨眼频率差异"这种假特征(解码作弊)。

from mne.preprocessing import ICA

ica = ICA(n_components=p["ica_n_components"], random_state=42, max_iter="auto")
ica.fit(raw)     # 在连续数据上拟合(数据长, 分解比在epoch上更稳定)

# 找眼电成分: 对每个代理通道分别算相关, 取并集
eog_idx, eog_scores = [], {}
for ch in p["eog_proxy_channels"]:
    idx, scores = ica.find_bads_eog(raw, ch_name=ch, verbose=False)
    eog_idx.extend(idx)
    for i in idx:
        eog_scores[i] = float(np.abs(scores[i]))
eog_idx = sorted(set(eog_idx))

# ---- 打印: 判定结果与证据强度 ----
print(f"ICA 分解出 {ica.n_components_} 个成分")
print(f"判定为眼电的成分: {eog_idx}")
for i in eog_idx:
    print(f"  成分{i}: 与额区代理通道的相关强度 = {eog_scores[i]:.2f}")
try:    # 新版MNE才有: 各成分解释的方差占比
    var_ratio = ica.get_explained_variance_ratio(raw)
    print(f"全部成分共解释方差: {var_ratio['eeg']*100:.1f}%")
except Exception:
    pass

# 成分地形图: 肉眼核对——被标记的成分应聚焦额区最前部
ica.plot_components(range(min(12, p["ica_n_components"])));

In [ ]:
# ============ Step 3b: 应用 ICA 并验证效果 ============
# 【目的】plot_overlay 直观对比"去除眼电成分前后"的信号:
#        额区通道(Fp1/Fp2)的大尖峰应明显消失, 其余通道基本不变
#        ——"只动了该动的"才说明成分选对了。

ica.plot_overlay(raw, exclude=eog_idx, picks="eeg");   # 红=原始, 黑=清洗后

ica.exclude = eog_idx
ica.apply(raw)          # 从信号中减去被标记成分, 重建其余部分

print(f"已去除 {len(eog_idx)} 个眼电成分, 信号重建完成")
print(f"清洗后幅值范围: [{raw.get_data(picks='eeg').min()*1e6:.0f}, "
      f"{raw.get_data(picks='eeg').max()*1e6:.0f}] uV (极端大幅值应有收敛)")

## Step 4 Epoch 切分

从"连续数据"切换到"试次 × 通道 × 时间"的三维结构——
后面所有分析(ERD/CSP/深度学习)都建立在这个结构上。

📚 **延伸干货**
- 📄 [MNE Epochs 官方教程](https://mne.tools/stable/auto_tutorials/epochs/10_epochs_overview.html)——Epoch 对象的完整操作手册
- 📄 [Autoreject 文档](https://autoreject.github.io/)——比固定阈值更智能的自适应试次剔除库(进阶可替换本步骤的 reject)

In [ ]:
# ============ Step 4: 按事件切分 Epoch ============
#
# 【方法】以每次"想象开始"的事件标记为时间零点, 向前取1s、向后取4s。
#        对齐的是"试次之间"的时间轴(通道之间本来就是同步采样的)。
#
# 【为什么这样设计】两个 MI 任务特有的决策:
#        ①baseline=None —— ERP分析必做基线校正(减去刺激前均值, 对齐电位零点),
#          但 MI 的特征是"频带功率的相对变化"不是电位幅值, 基线校正在
#          时频分析阶段做(见Step 5的 percent 模式), 这里不做。
#        ②reject=350uV —— 幅值超阈值的试次整个丢弃(ICA漏网的大伪迹兜底)。
#          同类方法: Autoreject库能逐通道自适应阈值+局部修复, 更精细但慢, 
#          教学场景固定阈值足够。
#
# 【目的】-1~0s 留作 ERD 基线窗; 0~4s 覆盖整个想象期。

events, event_map = mne.events_from_annotations(raw)

# ---- 打印: 事件统计 ----
print(f"注释->事件码映射: {event_map}")        # T0->1(静息) T1->2(左手) T2->3(右手)
uniq, cnt = np.unique(events[:, -1], return_counts=True)
for u, c in zip(uniq, cnt):
    name = {1: "T0静息", 2: "T1左手", 3: "T2右手"}.get(u, "?")
    print(f"  事件{u}({name}): {c} 次")

event_id = dict(cfg["data"]["event_id"])       # {'left': 2, 'right': 3}
epochs = mne.Epochs(raw, events, event_id,
                    tmin=p["epoch_tmin"], tmax=p["epoch_tmax"],
                    baseline=None, preload=True,
                    reject=dict(eeg=p["reject_eeg"]))

# ---- 打印: 切分结果与丢弃明细 ----
n_dropped = len([d for d in epochs.drop_log if len(d) > 0 and "IGNORED" not in d])
print(f"\nEpoch形状 (试次, 通道, 时间点): {epochs.get_data(copy=True).shape}")
print(f"保留: left={len(epochs['left'])}, right={len(epochs['right'])} "
      f"| 因超幅值丢弃: {n_dropped} 个试次")
print(f"每个试次时间轴: {epochs.tmin}s ~ {epochs.tmax}s "
      f"({len(epochs.times)} 个采样点)")

## Step 5 ERD 验证（解码前的质量门禁）

**先验证信号里真的存在神经生理效应，再谈解码**——否则解码分数再高
也可能是在学伪迹。验收标准（对侧支配）：

- 左手想象 → **C4**(右脑) mu/beta 功率下降（时频图蓝色）
- 右手想象 → **C3**(左脑) 功率下降
- 地形图上 ERD 聚焦中央区；全头弥漫 = 伪迹没除净

📚 **延伸干货**
- 📄 [Pfurtscheller & Lopes da Silva 1999](https://doi.org/10.1016/S1388-2457(99)00141-8)——ERD/ERS 奠基综述, 本步骤一切结论的原始出处
- 🎬 [YouTube: Mike X Cohen ANTS 时频分析播放列表](https://www.youtube.com/playlist?list=PLn0OLiymPak2BYu--bR0ADNBJsC4kuRWs)——Morlet 小波从直觉到数学讲得最好的课程(《Analyzing Neural Time Series》作者)
- 📄 [MNE 时频分析教程](https://mne.tools/stable/auto_tutorials/time-freq/20_sensors_time_frequency.html)——compute_tfr 的官方示例

In [ ]:
# ============ Step 5a: Morlet 小波时频变换 ============
#
# 【方法】ERD 是"感应(induced)"活动: mu/beta 振荡的相位不锁定于事件,
#        直接时域平均会正负抵消(这正是ERP看不到它的原因)。
#        必须"逐试次算功率 -> 再跨试次平均"。Morlet 小波 = 高斯包络的复正弦,
#        在每个(频率,时刻)点卡尺寸自适应的窗: n_cycles=freq/2 让低频用长窗(频率分辨率优先)、
#        高频用短窗(时间分辨率优先)。
#
# 【为什么选它】同类比较:
#        ①STFT(短时傅里叶)——固定窗长, 低频高频不能兼顾
#        ②Hilbert变换——需先窄带滤波, 适合单频段追踪, 不适合全频段扫描
#        ③多锥度(multitaper)——方差更小但计算重
#        Morlet 是时频分析的社区默认选择, MNE/FieldTrip 教程标配。
#
# 【基线模式 percent】(功率-基线均值)/基线均值, 即"相对基线变化了百分之几",
#        消掉了1/f背景谱, ERD直接读作负百分比。

e = cfg["erd"]
freqs = np.arange(e["freq_min"], e["freq_max"], 1)
n_cycles = freqs / 2.0

def tfr_of(cond):
    try:                                    # MNE >= 1.7 的新 API
        t = epochs[cond].compute_tfr("morlet", freqs=freqs, n_cycles=n_cycles,
                                     return_itc=False, average=True)
    except (AttributeError, TypeError):     # 旧版 API
        from mne.time_frequency import tfr_morlet
        t = tfr_morlet(epochs[cond], freqs=freqs, n_cycles=n_cycles,
                       return_itc=False)
    return t.apply_baseline(tuple(e["baseline"]), mode="percent")

# ---- 打印: 定量 ERD 数值表(想象期 0.5-3.5s, mu频段 8-13Hz 的平均变化) ----
print("定量ERD (mu频段功率相对基线的变化, 负值=抑制=皮层激活):")
print(f"{'':>10s} {'C3(左脑)':>12s} {'C4(右脑)':>12s}   期望模式")
for cond in ["left", "right"]:
    tfr = tfr_of(cond)
    vals = []
    for ch in ["C3", "C4"]:
        sub = tfr.copy().crop(tmin=0.5, tmax=3.5, fmin=8, fmax=13)
        vals.append(sub.data[sub.ch_names.index(ch)].mean() * 100)
    expect = "C4更负(对侧)" if cond == "left" else "C3更负(对侧)"
    print(f"{cond+'手想象':>10s} {vals[0]:>+11.1f}% {vals[1]:>+11.1f}%   {expect}")

# ---- 时频图: 2x2 矩阵(行=想象侧, 列=电极) ----
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for i, cond in enumerate(["left", "right"]):
    tfr = tfr_of(cond)
    for j, ch in enumerate(e["plot_channels"]):
        tfr.plot(picks=[ch], axes=axes[i, j], colorbar=(j == 1), show=False)
        expect = " <-- 期望ERD" if (cond, ch) in [("left", "C4"), ("right", "C3")] else ""
        axes[i, j].set_title(f"{cond} @ {ch}{expect}")
fig.suptitle(f"S{SUBJECT:03d} ERD: 对侧 mu/beta 功率应下降(蓝色)")
fig.tight_layout()

In [ ]:
# ============ Step 5b: ERD 地形图 ============
# 【目的】时频图只看了C3/C4两个点, 地形图检查全头空间分布:
#        ERD 聚焦中央区(运动皮层上方) = 生理效应;
#        全头弥漫或聚焦额区/边缘 = 伪迹残留, 应回头检查ICA。

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, cond in enumerate(["left", "right"]):
    tfr_of(cond).plot_topomap(tmin=0.5, tmax=3.5, fmin=8, fmax=13,
                              axes=axes[i], show=False)
    axes[i].set_title(f"{cond}想象 mu-ERD地形")
fig.tight_layout()
print("验收: 蓝色(功率下降)应集中在中央区, 且左右手想象呈对侧不对称")

## Step 6 CSP + LDA 解码

链路全景（训练学到的只有 CSP 的 W 矩阵 384 个数 + LDA 的 7 个数）：

```
(45试次,64通道,480点) --窄带8-30Hz--> --CSP投影+log方差--> (45,6) --LDA--> left/right
```

📚 **延伸干货**
- 🎬 [B站: 共空间模式算法详解(CSP)](https://www.bilibili.com/video/BV1AX4y187Li)——新手向, 配合下面的推导博客食用
- 📄 [共空间模式CSP 中文推导](https://mrswolf.github.io/common-spatial-pattern/)——从协方差到广义特征值的完整数学推导
- 📄 [Blankertz 2008: Optimizing Spatial Filters](https://doi.org/10.1109/MSP.2008.4408441)——CSP 领域"圣经"论文, 面试被问 CSP 细节看这篇
- 📄 [MNE 官方 CSP+LDA 解码示例](https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html)——用的正是本 notebook 同款 EEGBCI 数据集, 可交叉对照
- 📄 [EEGNet 论文 (Lawhern 2018)](https://arxiv.org/abs/1611.08024)——批量版模块4的模型出处, 图2的结构图值得细看

In [ ]:
# ============ Step 6a: 特征提取准备(窄带滤波 + 时间裁剪) ============
#
# 【方法-窄带滤波】EEG功率谱呈1/f形状, 宽带信号的总方差被低频成分主导。
#        CSP 的核心假设是"方差=你关心的频带功率", 只有先窄化到 8-30Hz(mu+beta),
#        这个等式才成立——否则CSP会去优化低频的类间差异(与任务无关)。
#
# 【方法-时间裁剪 0.5~3.5s】ERD动态: 0~0.5s还在建立, 3.5s后想象结束
#        会出现beta反弹(ERS, 功率反向飙升)——把ERS混进特征窗会稀释类间差异。
#
# 【目的】让"方差"这个数学量严格等于"想象期mu/beta功率"这个生理量。

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import ConfusionMatrixDisplay, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline

from src.decode_csp import get_xy

d = cfg["decode"]
X, y = get_xy(epochs, d["band"], d["crop"])   # 复用批量版同一函数

# ---- 打印: 特征提取前后的数据形态 ----
print(f"窄带({d['band'][0]}-{d['band'][1]}Hz) + 裁窗({d['crop'][0]}~{d['crop'][1]}s)后:")
print(f"  X形状 (试次,通道,时间点): {X.shape}")
print(f"  y标签分布: left={np.sum(y==0)}, right={np.sum(y==1)}")

# 演示"窄带滤波让方差有意义": 对比宽带/窄带下C3通道的类间方差比
X_wide, _ = get_xy(epochs, [1, 45], d["crop"])
c3 = epochs.ch_names.index("C3")
for name, XX in [("宽带1-45Hz", X_wide), ("窄带8-30Hz", X)]:
    v_left = XX[y == 0, c3].var(axis=1).mean()
    v_right = XX[y == 1, c3].var(axis=1).mean()
    print(f"  {name}: C3方差 left/right = {v_left/v_right:.2f} "
          f"(离1越远类间差异越清晰)")

In [ ]:
# ============ Step 6b: CSP+LDA 交叉验证 ============
#
# 【方法-CSP】共空间模式: 解广义特征值问题 Σ_left·w = λ·Σ_right·w,
#        找到投影方向使"左手方差/右手方差"最大(和最小), 各取3个方向共6个。
#        每个试次投影成6个虚拟通道, 取log方差 -> 6维特征向量。
#        取log的原因: 功率类量的分布右偏, log后近似高斯, 匹配LDA的分布假设。
#
# 【方法-LDA】线性判别分析: 在6维特征空间找方向使"类间散度/类内散度"最大,
#        然后放一个线性边界。与CSP是同一个数学模板(最大化方差比+特征分解),
#        一个用于提特征、一个用于分类, 全程无梯度下降——小样本稳定的根源。
#
# 【为什么选它】同类比较:
#        ①CSP+LDA —— 90年代至今的必报基线, 商用在线系统仍在部署(计算量小/可解释)
#        ②FBCSP —— 多子带CSP拼接, BCI Competition IV冠军方案(见批量版模块3)
#        ③黎曼几何 —— 2015年后学术强基线(pyriemann)
#        ④EEGNet等深度模型 —— 大数据/跨被试占优, 单被试45试次常打不过CSP
#
# 【防泄露】CSP是有监督的! 必须放进Pipeline在每个CV fold内部拟合,
#        如果先在全量数据上fit再交叉验证, 测试信息就泄露了, 分数虚高。

# 【坑-秩亏】平均参考(-1维)+ICA删眨眼成分(-1维)+坏导插值 使数据实际秩<64,
#        类协方差矩阵奇异, 广义特征值分解在部分fold报"not positive definite"。
#        reg="ledoit_wolf": 协方差收缩估计, 把方差为0的空方向垫成正定, 社区标准做法。
clf = Pipeline([("csp", CSP(n_components=d["csp_components"], reg="ledoit_wolf", log=True)),
                ("lda", LinearDiscriminantAnalysis())])
cv = StratifiedKFold(d["cv_folds"], shuffle=True, random_state=d["random_state"])

# ---- 打印: 每折成绩 + 汇总 ----
fold_accs = cross_val_score(clf, X, y, cv=cv)
print("各折准确率:", np.round(fold_accs, 3),
      f"| 均值 {fold_accs.mean():.3f} ± {fold_accs.std():.3f}")

y_pred = cross_val_predict(clf, X, y, cv=cv)
acc = float(np.mean(y_pred == y))
kappa = float(cohen_kappa_score(y, y_pred))
print(f"离线CV汇总: acc={acc:.3f}  kappa={kappa:.3f}  (随机水平: acc=0.5, kappa=0)")
print("kappa的意义: 扣除随机蒙对的部分, 0.6以上算好被试, 0.2以下接近BCI文盲")

ConfusionMatrixDisplay.from_predictions(y, y_pred, display_labels=["left", "right"]);

In [ ]:
# ============ Step 6c: 看看模型内部——特征与空间模式 ============
# 【目的】可解释性验证: ①6维特征在两类间是否真的分开
#        ②CSP空间模式是否聚焦C3/C4(=学到生理特征而非伪迹的直接证据)
# 注意: 这里为了可视化在全量数据上fit一次, 仅用于展示, 不用于报告成绩。

csp_vis = CSP(n_components=d["csp_components"], reg="ledoit_wolf", log=True)
feats = csp_vis.fit_transform(X, y)

# ---- 打印: 特征矩阵与类间分离度 ----
print(f"CSP特征矩阵形状: {feats.shape} (每试次: 480个时间点 -> 6个log功率)")
print("\n前3个试次的特征向量:")
for i in range(3):
    lab = "left " if y[i] == 0 else "right"
    print(f"  试次{i}({lab}): {np.round(feats[i], 2)}")
print("\n各特征维度的类间均值差(绝对值越大区分度越高):")
diff = np.abs(feats[y == 0].mean(0) - feats[y == 1].mean(0))
print(" ", np.round(diff, 2), "<- 首末成分应最大(CSP按判别力排序)")

# LDA在特征上的权重(哪些虚拟通道说了算)
lda_vis = LinearDiscriminantAnalysis().fit(feats, y)
print("\nLDA权重:", np.round(lda_vis.coef_[0], 2), f" 截距: {lda_vis.intercept_[0]:.2f}")

# 空间模式地形图: 头两个pattern应分别聚焦C3/C4附近
csp_vis.plot_patterns(epochs.info, components=range(4), colorbar=False, size=1.3);
print("验收: 前两个pattern应聚焦中央区左右两侧 —— 算法从数据里重新发现了对侧支配")

## Step 7 mini 伪在线：因果约束下还剩多少

离线 CV 分数**不代表**在线可用，两个关键差别：
1. **因果滤波**——在线只能看过去，`filtfilt` 换成 `sosfilt`（有群延迟）
2. **按时间切分**——前 70% 试次训练、后 30% 流式测试，不能 shuffle

训练也在因果滤波数据上做（**训练/推理口径一致**——和推荐系统离线特征
必须与在线 serving 对齐是同一个原则）。

📚 **延伸干货**
- 📄 [Wolpaw 2002 BCI 综述](https://doi.org/10.1016/S1388-2457(02)00057-3)——ITR 公式的原始出处, BCI 领域引用最高的论文之一
- 📄 [Lotte 2018 解码算法十年综述](https://doi.org/10.1088/1741-2552/aab2f2)——CSP/黎曼/深度学习三大门派的全景地图
- 📄 [MetaBCI 开源平台](https://github.com/TBC-TJU/MetaBCI)——天津大学开源的国产 BCI 全流程框架, 含真·在线模块, 伪在线之后的下一站

In [ ]:
# ============ Step 7: 伪在线流式评估 ============
#
# 【方法】把连续数据当作实时流重放: 整段做因果IIR带通(sosfilt, 只用过去样本),
#        前70%试次训练CSP+LDA, 后30%试次在不同"决策时刻"(想象开始后offset秒)
#        取长2s的滑窗盲判。扫描offset得到延迟-精度曲线。
#
# 【为什么这样设计】同类比较——评估在线可用性的三种方式:
#        ①真在线实验(金标准)——需要被试现场闭环, 成本高
#        ②伪在线(本方法)——用离线数据模拟因果约束, 行业标准的预检手段
#        ③离线CV直接外推——错误! 零相位滤波+shuffle切分都用了在线不存在的信息
#
# 【目的】回答三个在线核心问题: 性能存活多少? 多快能出结果? (批量版模块5
#        还会算静息期误触发率和ITR)

from src.pseudo_online import (causal_bandpass, compute_itr, fit_online_model,
                               latency_accuracy_curve, split_trials)

sfreq = raw.info["sfreq"]
stream = causal_bandpass(raw.get_data(picks="eeg"), sfreq, d["band"])

train_ev, test_ev = split_trials(events, event_id,
                                 cfg["pseudo_online"]["train_ratio"])
print(f"按时间切分: 前{len(train_ev)}试次训练 | 后{len(test_ev)}试次流式测试(不shuffle!)")

csp_on, lda_on = fit_online_model(stream, sfreq, train_ev, event_id, cfg)
curve = latency_accuracy_curve(stream, sfreq, test_ev, event_id,
                               csp_on, lda_on, cfg)

# ---- 打印: 延迟-精度明细表 ----
print("\n延迟-精度曲线(决策时刻 -> 流式准确率):")
print(curve.to_string(index=False,
                      formatters={"offset": "{:.2f}s".format,
                                  "acc": "{:.3f}".format}))

best = curve.loc[curve.acc.idxmax()]
itr = compute_itr(best.acc, 2, best.offset)
print(f"\n最佳决策时刻: {best.offset:.2f}s | acc={best.acc:.3f} "
      f"| ITR={itr:.1f} bits/min")
print(f"离线CV acc={acc:.3f} -> 伪在线best acc={best.acc:.3f} "
      f"(掉点来源: 因果滤波群延迟 + 时间切分 + 训练数据只有70%)")

# ---- 曲线图 ----
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve.offset, curve.acc, "o-")
ax.axhline(0.5, ls="--", c="gray", label="随机水平")
ax.axhline(acc, ls="--", c="g", label=f"离线CV acc={acc:.2f}")
ax.axvline(best.offset, ls=":", c="r",
           label=f"最佳 {best.offset:.2f}s acc={best.acc:.2f}")
ax.set_xlabel("决策时刻(想象开始后, 秒)")
ax.set_ylabel("流式准确率")
ax.set_title(f"S{SUBJECT:03d} 延迟-精度曲线(因果滤波+时间切分)")
ax.legend();

## 总结

| 步骤 | 关键决策 | 为什么 |
|---|---|---|
| 坏导检测 | 对数方差z分数 | 批量场景不能靠肉眼, 需要自动化+可复现 |
| 插值→重参考 | 顺序固定 | 坏导会拉偏平均参考 |
| ICA | 沿通道轴拆源 | 眼电与脑电空间混叠、频率重叠, 滤波去不掉 |
| Epoch | 不做基线校正 | MI 特征是频带功率, 不是电位 |
| ERD 验证 | 逐试次功率再平均 | 感应活动相位不锁定, 时域平均会抵消 |
| CSP 前窄带滤波 | 8–30 Hz | 让"方差"重新等于"关心频段的功率"(对齐优化目标) |
| CSP 窗口 | 0.5–3.5 s | 避开想象结束后的 beta 反弹(ERS) |
| CSP 特征取 log | log(方差) | 功率分布右偏, log后近似高斯, 匹配LDA假设 |
| 伪在线 | 因果滤波+时间切分 | 离线分数 ≠ 在线可用, 训练/推理口径必须一致 |

**自检问题**（能不看上文答出来才算掌握）：
1. 为什么 ERD 用时域平均看不到，ERP 却可以？
2. CSP 的"方差最大化"和 ERD 是什么关系？为什么必须先窄带滤波？
3. 伪在线掉点的三个来源是什么？
4. 如果 ERD 地形图全头弥漫，最可能是哪一步出了问题？
5. CSP 特征为什么取 log？

**下一步**：换 `SUBJECT` 观察个体差异；或运行批量版
`python -m src.preprocess` → `validate_erd` → `decode_csp` → `decode_eegnet`
→ `pseudo_online` → `report`，看 20 个被试的统计结论。

## 📚 资源汇总（按学习优先级排序）

| 优先级 | 资源 | 类型 | 对应步骤 |
|---|---|---|---|
| ⭐⭐⭐ | [MNE 滤波原理长文](https://mne.tools/stable/auto_tutorials/preprocessing/25_background_filtering.html) | 文档 | Step 2 |
| ⭐⭐⭐ | [CSP 中文推导博客](https://mrswolf.github.io/common-spatial-pattern/) + [B站视频](https://www.bilibili.com/video/BV1AX4y187Li) | 博客+视频 | Step 6 |
| ⭐⭐⭐ | [MNE 官方 CSP 解码示例](https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html)（同款数据集） | 代码 | Step 6 |
| ⭐⭐ | [Mike X Cohen 时频分析课](https://www.youtube.com/playlist?list=PLn0OLiymPak2BYu--bR0ADNBJsC4kuRWs) | 视频 | Step 5 |
| ⭐⭐ | [Python脑电处理中文手册](https://github.com/ZitongLu1996/Python-EEG-Handbook-CN) | 手册 | 全程 |
| ⭐⭐ | [EEGNet 论文](https://arxiv.org/abs/1611.08024) + [B站 BCI IV-2a 实战](https://www.bilibili.com/video/BV1BC41147QT) | 论文+视频 | 批量版模块4 |
| ⭐⭐ | [Delorme ICA 播放列表](https://www.youtube.com/playlist?list=PLXc9qfVbMMN2uDadxZ_OEsHjzcRtlLNxc) + [ICLabel 练习](https://labeling.ucsd.edu/tutorial) | 视频+互动 | Step 3 |
| ⭐ | [Pfurtscheller 1999 ERD综述](https://doi.org/10.1016/S1388-2457(99)00141-8) / [Blankertz 2008 CSP](https://doi.org/10.1109/MSP.2008.4408441) / [Wolpaw 2002](https://doi.org/10.1016/S1388-2457(02)00057-3) / [Lotte 2018](https://doi.org/10.1088/1741-2552/aab2f2) | 经典论文 | 深挖用 |
| ⭐ | [MetaBCI](https://github.com/TBC-TJU/MetaBCI) | 开源平台 | 伪在线的下一站 |

**使用建议**：先跑通本 notebook → 精读两篇 ⭐⭐⭐ 的文档/博客 → 论文按需查阅（面试前重点过 Blankertz 2008 和 Lotte 2018）。